%md
## Bronze Layer: Raw Ingestion
Load raw CSVs into Delta tables with no cleaning or type casting — everything read as string so dirty data doesn't break the job. Adds audit columns: `source_file`, `ingestion_ts`, `load_type`.

In [0]:
%run "./00_setup"

%md
### Step 0: Setup
Creating catalog, schemas (raw/silver/gold), and a volume to store raw files.

Catalog and schemas created successfully


### Batch Ingestion
Reading the 4 historical batch CSVs, adding audit columns, and writing each as a Delta table in the `raw` schema.

In [0]:
from pyspark.sql import functions as F

def add_audit_cols(df, load_type):
    return (
        df.withColumn("source_file", F.col("_metadata.file_path"))
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("load_type", F.lit(load_type))
    )

In [0]:
batch_files = {
    "bronze_orders": "orders_batch.csv",
    "bronze_customers": "customers_batch.csv",
    "bronze_products": "products_batch.csv",
    "bronze_stores": "stores_batch.csv",
}

for table_name, file_name in batch_files.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{BATCH_PATH}/{file_name}")
    )
    df = add_audit_cols(df, "batch")
    full_name = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"{full_name}: {df.count()} rows loaded")

retail_demo.raw.bronze_orders: 12180 rows loaded
retail_demo.raw.bronze_customers: 2560 rows loaded
retail_demo.raw.bronze_products: 830 rows loaded
retail_demo.raw.bronze_stores: 80 rows loaded


### Incremental Ingestion (Auto Loader)
Using `cloudFiles` to auto-detect new daily files without manually tracking what's already been processed. Schema evolution is enabled to handle the new `coupon_code` column that appears on day 3.

In [0]:
incremental_streams = {
    "bronze_orders_incremental": "orders_incremental_*.csv",
    "bronze_customers_cdc": "customers_cdc_*.csv",
    "bronze_products_cdc": "products_cdc_*.csv",
}

for table_name, pattern in incremental_streams.items():
    stream_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", True)
        .option("cloudFiles.schemaLocation", f"{SCHEMA_PATH}/{table_name}")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("pathGlobFilter", pattern)
        .load(INCR_PATH)
    )
    stream_df = add_audit_cols(stream_df, "incremental")

    full_name = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
    query = (
        stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{CHECKPOINT_PATH}/{table_name}")
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(full_name)
    )
    query.awaitTermination()
    print(f"{full_name} done ->", spark.table(full_name).count(), "rows")

retail_demo.raw.bronze_orders_incremental done -> 6135 rows
retail_demo.raw.bronze_customers_cdc done -> 630 rows
retail_demo.raw.bronze_products_cdc done -> 180 rows


### Sanity Check
Preview a few rows from each bronze table to confirm data looks right and audit columns are populated.

In [0]:
all_tables = list(batch_files.keys()) + list(incremental_streams.keys())
for t in all_tables:
    print(f"--- {t} ---")
    display(spark.table(f"{CATALOG}.{RAW_SCHEMA}.{t}").limit(3))

--- bronze_orders ---


order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,source_file,ingestion_ts,load_type
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/orders_batch.csv,2026-07-12T08:28:04.026Z,batch
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/orders_batch.csv,2026-07-12T08:28:04.026Z,batch
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/orders_batch.csv,2026-07-12T08:28:04.026Z,batch


--- bronze_customers ---


customer_id,customer_name,city,segment,gender,signup_date,status,source_file,ingestion_ts,load_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/customers_batch.csv,2026-07-12T08:28:13.911Z,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/customers_batch.csv,2026-07-12T08:28:13.911Z,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,dbfs:/Volumes/retail_demo/raw/retail_files/datasets/batch/customers_batch.csv,2026-07-12T08:28:13.911Z,batch


--- bronze_products ---


...

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,ingest_date,coupon_code,_rescued_data,source_file,ingestion_ts,load_type
OI3000001,2026-04-26 22:42:00,C01830,P00296,S056,5,64201.56,0.1,256806.24,CARD,returned,2026-04-26,null,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-12T08:30:15.215Z,incremental
OI3000002,2026-04-26 04:25:00,C01705,P00334,S030,2,16032.79,0.0,25652.46,NETBANKING,cancelled,2026-04-26,null,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-12T08:30:15.215Z,incremental
OI3000003,2026-04-23 00:00:00,C01378,P00014,S045,6,24907.63,0.1,141973.49,CARD,shipped,2026-04-26,NEW10,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv,2026-07-12T08:30:15.215Z,incremental


--- bronze_customers_cdc ---


customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,_rescued_data,source_file,ingestion_ts,load_type
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,not_a_date,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-12T08:30:27.729Z,incremental
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-12T08:30:27.729Z,incremental
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2026-07-12T08:30:27.729Z,incremental


--- bronze_products_cdc ---


product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation,_rescued_data,source_file,ingestion_ts,load_type
P00173,Speaker 173,Electronics,BrandA,50560.38,discontinued,2025-09-11,2026-04-25,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv,2026-07-12T08:30:38.875Z,incremental
P00680,Perfume 680,Beauty,BrandC,40368.75,active,2024-12-24,2026-04-25,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv,2026-07-12T08:30:38.875Z,incremental
P00095,Smartphone 95,Electronics,BrandB,685.97,active,2023-10-15,2026-04-25,UPDATE,null,/Volumes/retail_demo/raw/retail_files/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv,2026-07-12T08:30:38.875Z,incremental
